In [ ]:
# Preface 分块 + LangSmith/OpenAI Key 配置（换成真实 key；需在导入 langchain 前设置）
#Preface: Chunking
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = "<your-api-key>"
os.environ['OPENAI_API_KEY'] = "<your-api-key>"

In [ ]:
#Part 12: Multi-representation Indexing

In [ ]:
# Part 12 多表示索引：加载两篇博客作为原始文档
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

loader = WebBaseLoader("https://lilianweng.github.io/posts/2024-02-05-human-data-quality/")
docs.extend(loader.load())

In [ ]:
# 用 LLM 给每篇文档生成“摘要”。思路：用摘要做向量检索，命中后返回对应的完整原文
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{doc}")
    | ChatOpenAI(model="gpt-3.5-turbo",max_retries=0)
    | StrOutputParser()
)

summaries = chain.batch(docs, {"max_concurrency": 5})

In [ ]:
# 多向量检索：摘要存向量库(vectorstore)、原文存 docstore，用 doc_id 关联两者
# 修红：1.x 中 InMemoryByteStore 迁到 langchain_core.stores，
#       MultiVectorRetriever 迁到 langchain_classic.retrievers.multi_vector
from langchain_core.stores import InMemoryByteStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever

# The vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="summaries",
                     embedding_function=OpenAIEmbeddings())

# The storage layer for the parent documents
store = InMemoryByteStore()
id_key = "doc_id"

# The retriever
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)
doc_ids = [str(uuid.uuid4()) for _ in docs]

# Docs linked to summaries
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

# Add
retriever.vectorstore.add_documents(summary_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

In [ ]:
# 直接在向量库里按摘要相似度检索：返回的是“摘要”子文档
query = "Memory in agents"
sub_docs = vectorstore.similarity_search(query,k=1)
sub_docs[0]

In [ ]:
# 通过 retriever 检索：命中摘要后自动返回其对应的“完整原文”
# 修红：retriever.get_relevant_documents 在 1.x 已移除，改用 retriever.invoke(query)
retrieved_docs = retriever.invoke(query)
retrieved_docs[0].page_content[0:500]

In [ ]:
#Part 13: RAPTOR

In [ ]:
#Part 14: ColBERT

In [ ]:
# Part 14 ColBERT / RAGatouille：加载预训练 ColBERT 模型
# 注意：需先 `pip install ragatouille`（会引入 torch/faiss 等重依赖），未安装此处会报红
from ragatouille import RAGPretrainedModel
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

In [ ]:
# 用 Wikipedia API 抓取整篇页面的纯文本，作为待索引语料
import requests

def get_wikipedia_page(title: str):
    """
    Retrieve the full text content of a Wikipedia page.

    :param title: str - Title of the Wikipedia page.
    :return: str - Full text content of the page as raw string.
    """
    # Wikipedia API endpoint
    URL = "https://en.wikipedia.org/w/api.php"

    # Parameters for the API request
    params = {
        "action": "query",
        "format": "json",
        "titles": title,
        "prop": "extracts",
        "explaintext": True,
    }

    # Custom User-Agent header to comply with Wikipedia's best practices
    headers = {"User-Agent": "RAGatouille_tutorial/0.0.1 (ben@clavie.eu)"}

    response = requests.get(URL, params=params, headers=headers)
    data = response.json()

    # Extracting page content
    page = next(iter(data["query"]["pages"].values()))
    return page["extract"] if "extract" in page else None

full_document = get_wikipedia_page("Hayao_Miyazaki")

In [ ]:
# 用 RAGatouille 对文档建 ColBERT 索引（按 180 长度切分）
RAG.index(
    collection=[full_document],
    index_name="Miyazaki-123",
    max_document_length=180,
    split_documents=True,
)

In [ ]:
# ColBERT 检索：返回 top-3 结果
results = RAG.search(query="What animation studio did Miyazaki found?", k=3)
results

In [ ]:
# 把 ColBERT 包装成 langchain 检索器后再检索
retriever = RAG.as_langchain_retriever(k=3)
retriever.invoke("What animation studio did Miyazaki found?")